# 🤗 Modelos Preentrenados con Hugging Face

*Cuaderno de estudio diseñado por Matías Barreto, 2025*

## Introducción

En este cuaderno vamos a explorar el mundo de los **modelos preentrenados de última generación** usando la biblioteca Hugging Face Transformers. 

### ¿Qué son los modelos preentrenados?

Los modelos preentrenados son redes neuronales que ya fueron entrenadas en enormes conjuntos de datos por equipos de investigación de todo el mundo. En lugar de entrenar desde cero (lo que requeriría semanas de computación y miles de dólares), podemos usar estos modelos directamente.

### ¿Por qué usar Hugging Face?

Hugging Face es la plataforma líder para modelos de IA, que ofrece:
- **Acceso fácil** a miles de modelos estado del arte
- **APIs simples** que funcionan con pocas líneas de código
- **Documentación excelente** y comunidad activa
- **Modelos en español** y muchos otros idiomas

---

## Objetivos de Aprendizaje

Al finalizar este cuaderno, vas a poder:

1. **Cargar y usar modelos preentrenados** de Hugging Face
2. **Clasificar imágenes** con Vision Transformer (ViT)
3. **Realizar clasificación zero-shot** con CLIP usando lenguaje natural
4. **Detectar objetos múltiples** en imágenes con DETR
5. **Comparar diferentes arquitecturas** y sus casos de uso
6. **Trabajar con imágenes locales** de tu entorno

---

## Modelos que vamos a explorar

| Modelo | Tarea | Característica Principal |
|--------|-------|-------------------------|
| **ViT** (Vision Transformer) | Clasificación de imágenes | Aplica Transformers a visión |
| **CLIP** (OpenAI) | Clasificación zero-shot | Entiende texto e imágenes |
| **DETR** (Facebook) | Detección de objetos | Localiza múltiples objetos |

---

## Configuración del Entorno

In [ ]:
# INSTALACIÓN DE DEPENDENCIAS
# Estas librerías nos van a permitir trabajar con modelos de Hugging Face

!pip install transformers torch pillow requests matplotlib

print("Instalación completada")

In [ ]:
# IMPORTAR LIBRERÍAS

import torch
from transformers import pipeline
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML
import os
from google.colab import files
import cv2

print("Librerías importadas correctamente")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers disponible")
print("\nListo para explorar modelos preentrenados!")

## Funciones Auxiliares para Manejo de Imágenes

In [ ]:
# FUNCIONES PARA CARGAR IMÁGENES

def cargar_imagen_local():
    """
    Permite al usuario subir una imagen desde su computadora
    Retorna: imagen PIL
    """
    print("Selecciona una imagen desde tu computadora:")
    uploaded = files.upload()
    
    # Obtener el nombre del archivo subido
    filename = list(uploaded.keys())[0]
    
    # Cargar la imagen
    image = Image.open(filename)
    print(f"Imagen cargada: {filename}")
    print(f"Dimensiones: {image.size}")
    
    return image

def cargar_imagen_url(url):
    """
    Carga una imagen desde una URL
    """
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))
        print(f"Imagen cargada desde URL")
        print(f"Dimensiones: {image.size}")
        return image
    except Exception as e:
        print(f"Error al cargar imagen: {e}")
        return None

def mostrar_resultados(image, results, title="Resultados"):
    """
    Muestra la imagen y los resultados de forma elegante
    """
    plt.figure(figsize=(14, 7))

    # Mostrar imagen
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')

    # Mostrar resultados
    plt.subplot(1, 2, 2)
    plt.axis('off')

    # Crear texto de resultados
    result_text = ""
    for i, result in enumerate(results[:5], 1):
        if isinstance(result, dict):
            if 'label' in result and 'score' in result:
                # Crear barra de confianza visual
                confidence_bar = "█" * int(result['score'] * 20)
                result_text += f"{i}. {result['label']}\n"
                result_text += f"   Confianza: {result['score']:.1%} {confidence_bar}\n\n"

    plt.text(0.05, 0.95, result_text, fontsize=12, 
             verticalalignment='top', horizontalalignment='left',
             family='monospace', 
             bbox=dict(boxstyle='round,pad=1', facecolor='lightblue', alpha=0.7))

    plt.tight_layout()
    plt.show()

print("Funciones auxiliares definidas")

---

# MODELO 1: Vision Transformer (ViT)

## ¿Qué es Vision Transformer?

**Vision Transformer (ViT)** es un modelo revolucionario que aplica la arquitectura Transformer (originalmente diseñada para procesamiento de lenguaje natural) a la visión por computadora.

### Características Principales:

- **Arquitectura Transformer**: Usa mecanismos de atención en lugar de convoluciones
- **División en patches**: Divide la imagen en pequeños cuadrados que trata como "palabras"
- **Entrenado en ImageNet**: Reconoce 1000 categorías diferentes
- **Alta precisión**: Supera a muchas CNNs tradicionales

### ¿Cuándo usar ViT?

**Ideal para**: Clasificación general de imágenes  
**Ventajas**: Rápido, preciso, categorías amplias  
**Limitaciones**: Categorías fijas (no personalizable)

---

In [ ]:
# CARGAR MODELO ViT

print("Cargando Vision Transformer...")
print("(Esto puede tardar un momento la primera vez)")

# Crear el pipeline de clasificación
# Este pipeline descarga automáticamente el modelo y lo configura
vit_classifier = pipeline(
    "image-classification", 
    model="google/vit-base-patch16-224"
)

print("Vision Transformer cargado correctamente")
print("Modelo entrenado en ImageNet (1000 categorías)")
print("Listo para clasificar imágenes")

## Experimentando con ViT

In [ ]:
# EJEMPLO 1: IMAGEN DE EJEMPLO

print("Ejemplo 1: Clasificando una imagen de ejemplo")
print("=" * 50)

# Cargar imagen de ejemplo desde URL
ejemplo_url = "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=640"
imagen_ejemplo = cargar_imagen_url(ejemplo_url)

if imagen_ejemplo:
    print("\nProcesando imagen con ViT...")
    resultados_vit = vit_classifier(imagen_ejemplo)

    print("\nTop 5 Predicciones:")
    print("-" * 40)
    for i, resultado in enumerate(resultados_vit[:5], 1):
        print(f"{i}. {resultado['label']:25s} → {resultado['score']:.1%}")

    # Visualizar resultados
    mostrar_resultados(imagen_ejemplo, resultados_vit, "ViT - Clasificación de Imagen")

In [ ]:
# EJEMPLO 2: TU PROPIA IMAGEN

print("Ejemplo 2: Clasifica tu propia imagen")
print("=" * 50)

respuesta = input("¿Querés subir una imagen desde tu computadora? (s/n): ").lower()

if respuesta == 's':
    print("\nSubiendo imagen...")
    mi_imagen = cargar_imagen_local()
    
    print("\nClasificando tu imagen con ViT...")
    mis_resultados = vit_classifier(mi_imagen)
    
    print("\nResultados de tu imagen:")
    print("-" * 40)
    for i, resultado in enumerate(mis_resultados[:5], 1):
        print(f"{i}. {resultado['label']:25s} → {resultado['score']:.1%}")
    
    # Visualizar
    mostrar_resultados(mi_imagen, mis_resultados, "ViT - Tu Imagen")
else:
    print("Perfecto! Continuemos con el siguiente modelo.")

### Reflexión sobre ViT

**Preguntas para pensar:**
- ¿Qué tan precisas fueron las predicciones?
- ¿Reconoció objetos que no esperabas?
- ¿Cómo se comparan los resultados con lo que ves humanamente?

---

# MODELO 2: CLIP (Contrastive Language-Image Pre-training)

## ¿Qué es CLIP?

**CLIP** es un modelo revolucionario de OpenAI que conecta el mundo de las imágenes con el lenguaje natural. En lugar de estar limitado a categorías fijas, CLIP puede entender descripciones en texto libre.

### Características Principales:

- **Multimodal**: Entiende texto e imágenes conjuntamente
- **Zero-shot**: No necesita entrenamiento adicional para nuevas categorías
- **Lenguaje natural**: Podés usar tus propias descripciones
- **Flexible**: Adaptable a cualquier dominio específico

### ¿Cuándo usar CLIP?

**Ideal para**: Búsquedas específicas, categorías personalizadas  
**Ventajas**: Extremadamente flexible, entiende contexto  
**Limitaciones**: Puede ser menos preciso que modelos especializados

### La Magia de Zero-Shot

Con CLIP podés preguntarle cosas como:
- "Una persona usando anteojos de sol"
- "Un paisaje nocturno"
- "Comida italiana"
- "Un animal peligroso"

Sin haberlo entrenado específicamente para esas categorías!

---

In [ ]:
# CARGAR MODELO CLIP

print("Cargando CLIP...")
print("(Esto puede tardar un momento la primera vez)")

# Crear el pipeline de clasificación zero-shot
clip_classifier = pipeline(
    "zero-shot-image-classification", 
    model="openai/clip-vit-base-patch32"
)

print("CLIP cargado correctamente")
print("Listo para clasificación con lenguaje natural")
print("Ahora podés usar tus propias descripciones")

## Experimentando con CLIP

In [ ]:
# EJEMPLO 1: CLASIFICACIÓN CON DESCRIPCIONES PERSONALIZADAS

print("Ejemplo 1: CLIP con categorías personalizadas")
print("=" * 50)

# Cargar imagen de ejemplo
ejemplo_url_2 = "https://images.unsplash.com/photo-1505740420928-5e560c06d30e?w=640"
imagen_clip = cargar_imagen_url(ejemplo_url_2)

if imagen_clip:
    # AQUÍ ESTÁ LA MAGIA: Define tus propias categorías en lenguaje natural
    categorias_candidatas = [
        "una persona usando auriculares",
        "una persona sin auriculares", 
        "equipo de música",
        "un teléfono móvil",
        "una computadora portátil"
    ]

    print("\nCategorías que vamos a probar:")
    for i, categoria in enumerate(categorias_candidatas, 1):
        print(f"   {i}. {categoria}")

    print("\nProcesando con CLIP...")
    resultados_clip = clip_classifier(imagen_clip, candidate_labels=categorias_candidatas)

    print("\nResultados de CLIP:")
    print("-" * 60)
    for i, resultado in enumerate(resultados_clip, 1):
        barra_visual = "█" * int(resultado['score'] * 30)
        print(f"{i}. {resultado['label']:30s} → {resultado['score']:.1%} {barra_visual}")

    # Visualizar
    mostrar_resultados(imagen_clip, resultados_clip, "CLIP - Zero-Shot Classification")

In [ ]:
# EJEMPLO 2: CLIP CON TUS PROPIAS CATEGORÍAS

print("Ejemplo 2: Creá tus propias categorías")
print("=" * 50)

respuesta = input("¿Querés definir tus propias categorías para clasificar? (s/n): ").lower()

if respuesta == 's':
    print("\nDefiní 3-5 categorías separadas por comas")
    print("Ejemplos: 'un perro feliz', 'un gato durmiendo', 'un paisaje montañoso'")
    
    categorias_usuario = input("\nTus categorías: ").split(',')
    categorias_usuario = [cat.strip() for cat in categorias_usuario if cat.strip()]
    
    if len(categorias_usuario) >= 2:
        print(f"\nCategorías creadas: {len(categorias_usuario)}")
        for i, cat in enumerate(categorias_usuario, 1):
            print(f"   {i}. {cat}")
        
        # Elegir fuente de imagen
        print("\n¿De dónde querés obtener la imagen?")
        fuente = input("(l)ocal (subir archivo) / (u)rl (desde internet): ").lower()
        
        imagen_personalizada = None
        
        if fuente == 'l':
            imagen_personalizada = cargar_imagen_local()
        elif fuente == 'u':
            url_usuario = input("Ingresá la URL de la imagen: ")
            imagen_personalizada = cargar_imagen_url(url_usuario)
        
        if imagen_personalizada:
            print(f"\nClasificando con tus categorías...")
            resultados_personalizados = clip_classifier(
                imagen_personalizada, 
                candidate_labels=categorias_usuario
            )
            
            print("\nResultados:")
            print("-" * 50)
            for i, resultado in enumerate(resultados_personalizados, 1):
                barra = "█" * int(resultado['score'] * 25)
                print(f"{i}. {resultado['label']:25s} → {resultado['score']:.1%} {barra}")
            
            mostrar_resultados(imagen_personalizada, resultados_personalizados, "CLIP - Tus Categorías")
    else:
        print("Necesitás al menos 2 categorías para comparar")
else:
    print("Perfecto! Continuemos con el siguiente modelo.")

### Reflexión sobre CLIP

**Preguntas para pensar:**
- ¿Cómo se comparan los resultados de CLIP vs ViT?
- ¿Qué ventajas tiene poder usar lenguaje natural?
- ¿En qué situaciones preferirías CLIP sobre ViT?

---

# MODELO 3: DETR (Detection Transformer)

## ¿Qué es DETR?

**DETR (Detection Transformer)** es un modelo de detección de objetos desarrollado por Facebook AI. A diferencia de los modelos anteriores que clasifican toda la imagen, DETR puede detectar y localizar múltiples objetos dentro de una sola imagen.

### Características Principales:

- **Detección múltiple**: Encuentra varios objetos en una imagen
- **Localización espacial**: Proporciona coordenadas exactas (bounding boxes)
- **91 categorías COCO**: Entrenado en el famoso dataset COCO
- **Arquitectura Transformer**: Usa atención para detectar relaciones entre objetos

### ¿Cuándo usar DETR?

**Ideal para**: Escenas complejas con múltiples objetos  
**Ventajas**: Localización precisa, detecta relaciones espaciales  
**Limitaciones**: Más lento, limitado a categorías COCO

### Categorías COCO

DETR puede detectar 91 tipos de objetos diferentes, incluyendo:
- **Personas y animales**: person, cat, dog, horse, etc.
- **Vehículos**: car, motorcycle, airplane, bus, etc.
- **Objetos cotidianos**: chair, table, laptop, phone, etc.
- **Comida**: apple, banana, pizza, cake, etc.

---

In [ ]:
# CARGAR MODELO DETR

print("Cargando DETR...")
print("(Esto puede tardar un momento la primera vez)")

# Crear el pipeline de detección de objetos
detr_detector = pipeline(
    "object-detection", 
    model="facebook/detr-resnet-50"
)

print("DETR cargado correctamente")
print("Modelo entrenado en COCO (91 categorías)")
print("Listo para detectar múltiples objetos")

In [ ]:
# FUNCIÓN PARA DIBUJAR BOUNDING BOXES

def dibujar_detecciones(imagen, detecciones, umbral_confianza=0.7):
    """
    Dibuja las cajas delimitadoras en la imagen
    
    Args:
        imagen: Imagen PIL original
        detecciones: Lista de detecciones de DETR
        umbral_confianza: Confianza mínima para mostrar detección
    
    Returns:
        Imagen PIL con las detecciones dibujadas
    """
    # Crear copia de la imagen
    img_con_cajas = imagen.copy()
    draw = ImageDraw.Draw(img_con_cajas)
    
    # Colores para diferentes objetos
    colores = ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'pink', 'cyan']
    
    detecciones_validas = 0
    
    for i, deteccion in enumerate(detecciones):
        if deteccion['score'] >= umbral_confianza:
            box = deteccion['box']
            etiqueta = deteccion['label']
            confianza = deteccion['score']
            
            # Coordenadas de la caja
            xmin, ymin = int(box['xmin']), int(box['ymin'])
            xmax, ymax = int(box['xmax']), int(box['ymax'])
            
            # Color para esta detección
            color = colores[detecciones_validas % len(colores)]
            
            # Dibujar rectángulo
            draw.rectangle([xmin, ymin, xmax, ymax], outline=color, width=3)
            
            # Dibujar etiqueta con fondo
            texto = f"{etiqueta} {confianza:.2f}"
            
            # Calcular tamaño del texto para el fondo
            try:
                # Intentar usar una fuente por defecto
                font = ImageFont.load_default()
                bbox = draw.textbbox((0, 0), texto, font=font)
                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]
            except:
                # Fallback si hay problemas con la fuente
                text_width = len(texto) * 8
                text_height = 15
            
            # Dibujar fondo para el texto
            draw.rectangle([xmin, ymin-text_height-5, xmin+text_width+10, ymin], 
                         fill=color, outline=color)
            
            # Dibujar texto
            draw.text((xmin+2, ymin-text_height-2), texto, fill='white')
            
            detecciones_validas += 1
    
    return img_con_cajas, detecciones_validas

print("Función de visualización definida")

## Experimentando con DETR

In [ ]:
# EJEMPLO 1: DETECCIÓN EN IMAGEN DE EJEMPLO

print("Ejemplo 1: Detectando múltiples objetos")
print("=" * 50)

# Cargar imagen con múltiples objetos
ejemplo_url_3 = "https://images.unsplash.com/photo-1544568100-847a948585b9?w=640"
imagen_detr = cargar_imagen_url(ejemplo_url_3)

if imagen_detr:
    print("\nDetectando objetos con DETR...")
    resultados_detr = detr_detector(imagen_detr)
    
    print(f"\nDetectados {len(resultados_detr)} objetos!")
    print("-" * 60)
    
    for i, deteccion in enumerate(resultados_detr, 1):
        box = deteccion['box']
        print(f"{i:2d}. {deteccion['label']:15s} → Confianza: {deteccion['score']:.1%}")
        print(f"     Posición: ({int(box['xmin'])}, {int(box['ymin'])}) - ({int(box['xmax'])}, {int(box['ymax'])})")
        print(f"     Tamaño: {int(box['xmax']-box['xmin'])}×{int(box['ymax']-box['ymin'])} píxeles\n")
    
    # Dibujar las detecciones
    imagen_con_cajas, num_validas = dibujar_detecciones(imagen_detr, resultados_detr, umbral_confianza=0.7)
    
    # Mostrar comparación lado a lado
    plt.figure(figsize=(16, 8))
    
    plt.subplot(1, 2, 1)
    plt.imshow(imagen_detr)
    plt.title("Imagen Original", fontsize=14, fontweight='bold')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(imagen_con_cajas)
    plt.title(f"Detecciones DETR ({num_validas} objetos)", fontsize=14, fontweight='bold')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nEstadísticas de detección:")
    print(f"   • Total detectado: {len(resultados_detr)} objetos")
    print(f"   • Con alta confianza (>70%): {num_validas} objetos")
    print(f"   • Objetos únicos detectados: {len(set(d['label'] for d in resultados_detr))}")

In [ ]:
# EJEMPLO 2: DETECCIÓN EN TU PROPIA IMAGEN

print("Ejemplo 2: Detectá objetos en tu propia imagen")
print("=" * 50)

respuesta = input("¿Querés detectar objetos en una imagen tuya? (s/n): ").lower()

if respuesta == 's':
    print("\n¿De dónde querés obtener la imagen?")
    fuente = input("(l)ocal (subir archivo) / (u)rl (desde internet): ").lower()
    
    mi_imagen_detr = None
    
    if fuente == 'l':
        mi_imagen_detr = cargar_imagen_local()
    elif fuente == 'u':
        url_usuario = input("Ingresá la URL de la imagen: ")
        mi_imagen_detr = cargar_imagen_url(url_usuario)
    
    if mi_imagen_detr:
        print("\nDetectando objetos en tu imagen...")
        mis_detecciones = detr_detector(mi_imagen_detr)
        
        if mis_detecciones:
            print(f"\nDetectados {len(mis_detecciones)} objetos en tu imagen!")
            print("-" * 50)
            
            for i, deteccion in enumerate(mis_detecciones, 1):
                print(f"{i:2d}. {deteccion['label']:15s} → {deteccion['score']:.1%}")
            
            # Dibujar detecciones
            mi_imagen_con_cajas, mi_num_validas = dibujar_detecciones(
                mi_imagen_detr, mis_detecciones, umbral_confianza=0.5
            )
            
            # Mostrar resultados
            plt.figure(figsize=(16, 8))
            
            plt.subplot(1, 2, 1)
            plt.imshow(mi_imagen_detr)
            plt.title("Tu Imagen Original", fontsize=14, fontweight='bold')
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.imshow(mi_imagen_con_cajas)
            plt.title(f"Tus Detecciones ({mi_num_validas} objetos)", fontsize=14, fontweight='bold')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
        else:
            print("No se detectaron objetos en la imagen")
else:
    print("Perfecto! Pasemos a la comparación de modelos.")

### Reflexión sobre DETR

**Preguntas para pensar:**
- ¿Cómo se compara la información de DETR vs los modelos anteriores?
- ¿Qué ventajas tiene conocer la ubicación exacta de los objetos?
- ¿En qué aplicaciones del mundo real sería útil DETR?

---

# Comparación y Análisis de los Tres Modelos

## Tabla Comparativa

| Aspecto | ViT | CLIP | DETR |
|---------|-----|------|------|
| **Tarea Principal** | Clasificación de imagen completa | Clasificación zero-shot | Detección múltiple de objetos |
| **Arquitectura** | Vision Transformer | Transformer multimodal | Detection Transformer |
| **Flexibilidad** | Baja (1000 categorías fijas) | Muy alta (categorías libres) | Media (91 categorías COCO) |
| **Velocidad** | Muy rápida | Rápida | Moderada |
| **Precisión** | Muy alta | Alta | Alta en detección |
| **Información Espacial** | No (solo clasificación) | No (solo clasificación) | Sí (coordenadas exactas) |
| **Personalización** | No (categorías fijas) | Sí (lenguaje natural) | No (categorías COCO) |

---

In [ ]:
# COMPARACIÓN VISUAL DE LOS TRES MODELOS

print("COMPARACIÓN DE LOS TRES MODELOS")
print("=" * 60)

comparacion = """
┌────────────────────┬─────────────────────────────────────────────────────────┐
│ MODELO             │ CARACTERÍSTICAS PRINCIPALES                             │
├────────────────────┼─────────────────────────────────────────────────────────┤
│ ViT                │ • Clasificación general (1000 categorías ImageNet)      │
│ (Vision            │ • Arquitectura Transformer aplicada a visión           │
│  Transformer)      │ • Rápido y muy preciso para clasificación              │
│                    │ • Ideal para: identificar el objeto principal          │
├────────────────────┼─────────────────────────────────────────────────────────┤
│ CLIP               │ • Clasificación con descripciones de lenguaje natural   │
│ (OpenAI)           │ • Zero-shot: sin entrenamiento adicional                │
│                    │ • Categorías completamente personalizables              │
│                    │ • Ideal para: búsquedas específicas y flexibles        │
├────────────────────┼─────────────────────────────────────────────────────────┤
│ DETR               │ • Detección y localización de múltiples objetos         │
│ (Facebook)         │ • Proporciona coordenadas exactas (bounding boxes)      │
│                    │ • 91 categorías del dataset COCO                        │
│                    │ • Ideal para: análisis de escenas complejas             │
└────────────────────┴─────────────────────────────────────────────────────────┘
"""

print(comparacion)

print("\n¿CUÁNDO USAR CADA MODELO?")
print("-" * 40)
print("ViT:   Cuando necesitás clasificar UNA imagen en categorías generales")
print("CLIP:  Cuando querés buscar conceptos específicos o usar tus propias categorías")
print("DETR:  Cuando necesitás encontrar y localizar MÚLTIPLES objetos en una escena")

print("\nCASOS DE USO REALES:")
print("-" * 30)
print("App de fotos → ViT para organizar por categorías")
print("Buscador visual → CLIP para 'encontrá fotos de personas felices'")
print("Auto autónomo → DETR para detectar peatones, autos, señales")
print("Diagnóstico médico → CLIP para 'radiografía con fractura'")
print("Control de inventario → DETR para contar objetos en almacén")

---

# Ejercicios Propuestos

## Desafíos para Practicar

### Nivel Básico
1. **Comparación directa**: Tomá la misma imagen y pasala por los tres modelos. Compará los resultados.

2. **Especialización de CLIP**: Creá categorías específicas para tu área de interés (deportes, comida, arte, etc.)

### Nivel Intermedio  
3. **Detector de equipos de seguridad**: Usá CLIP para detectar "persona con casco", "persona sin casco", "persona con chaleco reflectante"

4. **Análisis de emociones**: Con CLIP, clasificá rostros como "persona feliz", "persona triste", "persona concentrada"

5. **Contador de objetos**: Usá DETR para contar cuántos objetos de cada tipo hay en una imagen

### Nivel Avanzado
6. **Sistema híbrido**: Combiná DETR + CLIP: 
   - Usá DETR para detectar personas
   - Para cada persona, usá CLIP para determinar "tiene barbijo" vs "no tiene barbijo"

7. **Análisis de escenas**: Creá un sistema que use los tres modelos para describir completamente una imagen

---

# Glosario de Términos

**Attention (Atención)**: Mecanismo que permite al modelo "enfocarse" en partes relevantes de la entrada.

**Bounding Box**: Rectángulo que enmarca un objeto detectado, definido por coordenadas (x_min, y_min, x_max, y_max).

**COCO Dataset**: Conjunto de datos con 91 categorías de objetos cotidianos, usado para entrenar modelos de detección.

**Fine-tuning**: Proceso de ajustar un modelo preentrenado para una tarea específica.

**Hugging Face**: Plataforma líder para compartir y usar modelos de IA, especialmente en NLP y visión.

**ImageNet**: Enorme base de datos de imágenes con 1000 categorías, estándar para entrenar modelos de visión.

**Multimodal**: Modelos que pueden procesar múltiples tipos de datos (texto, imágenes, audio).

**Object Detection**: Tarea de encontrar y localizar múltiples objetos en una imagen.

**Patch**: Pequeño fragmento rectangular de una imagen, usado por Vision Transformers.

**Pipeline**: Interfaz simplificada de Hugging Face que encapsula todo el proceso de procesamiento.

**Preentrenado**: Modelo que ya fue entrenado en un gran conjunto de datos y puede usarse directamente.

**Transformer**: Arquitectura de red neuronal basada en mecanismos de atención, originalmente para NLP.

**Zero-shot**: Capacidad de realizar tareas sin entrenamiento específico para esas tareas.

---

# Preguntas y Respuestas

## Preguntas Frecuentes

**P: ¿Por qué usar modelos preentrenados en lugar de entrenar desde cero?**  
R: Los modelos preentrenados ahorran tiempo, dinero y recursos computacionales enormes. Además, están entrenados en datos más diversos de los que la mayoría de personas puede acceder.

**P: ¿CLIP realmente "entiende" las descripciones de texto?**  
R: CLIP aprendió asociaciones entre texto e imágenes de millones de ejemplos. No "entiende" como los humanos, pero puede hacer conexiones muy sofisticadas.

**P: ¿Por qué DETR es más lento que ViT?**  
R: DETR tiene que procesar múltiples objetos y calcular sus ubicaciones exactas, lo cual es computacionalmente más intensivo que clasificar una sola imagen.

**P: ¿Puedo usar estos modelos para datos médicos o científicos?**  
R: Sí, pero tené cuidado. Los modelos están entrenados en imágenes "cotidianas". Para aplicaciones críticas como medicina, considerá modelos especializados.

**P: ¿Cómo mejoro la precisión de las predicciones?**  
R: 
- Usá imágenes de buena calidad
- Para CLIP, sé específico en las descripciones
- Considerá fine-tuning para tu dominio específico
- Combiná múltiples modelos para mayor robustez

**P: ¿Estos modelos funcionan en otros idiomas?**  
R: CLIP funciona razonablemente bien en español y otros idiomas. ViT y DETR son independientes del idioma ya que solo procesan imágenes.

---

# Resumen y Próximos Pasos

## Lo que aprendiste

En este cuaderno exploraste:

1. **Tres arquitecturas revolucionarias** de deep learning aplicadas a visión
2. **Uso práctico de Hugging Face** para acceder a modelos estado del arte
3. **Diferencias entre clasificación y detección** de objetos
4. **Poder del zero-shot learning** con CLIP
5. **Trabajo con imágenes locales** en lugar de solo URLs

## Conceptos Clave Dominados

- **Vision Transformers**: Aplicación de atención a imágenes
- **Modelos multimodales**: Conexión entre texto e imágenes  
- **Detección de objetos**: Localización espacial de múltiples elementos
- **APIs modernas**: Uso de pipelines para simplificar el trabajo

## Hacia Dónde Continuar

1. **Explorá más modelos** en [Hugging Face Hub](https://huggingface.co/models)
2. **Aprendé fine-tuning** para adaptar modelos a tus datos
3. **Combiná modelos** para crear sistemas más robustos
4. **Explorá otras modalidades**: audio, video, texto

## Recursos Adicionales

- **Hugging Face Course**: [huggingface.co/course](https://huggingface.co/course)
- **Papers with Code**: [paperswithcode.com](https://paperswithcode.com/)
- **Transformer Architecture**: Paper original "Attention Is All You Need"
- **CLIP Paper**: "Learning Transferable Visual Representations"

---

## Felicitaciones!

Completaste exitosamente tu introducción a modelos preentrenados de última generación. Ahora tenés las herramientas para:

- Usar modelos estado del arte sin entrenar desde cero
- Elegir el modelo correcto para cada tarea
- Trabajar con imágenes locales y personalizadas
- Entender las fortalezas y limitaciones de cada enfoque

**¡Seguí experimentando y construyendo proyectos increíbles!**